# Universal AC-MOT — T4 fidelity gate + official timing (Run all)
Checks out the commit named in `research/T4_TARGET_COMMIT.txt` (the V5 candidate), builds T4 caches of the fixed gate subset (no metrics), and runs the official T4 benchmark of that exact commit for the V4 and V5 policy files. Everything is written to Drive `Universal-ACMOT-Results/t4_gate/`.

In [ ]:
import subprocess, torch, os, shutil, json, hashlib, glob
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'needs a T4 runtime'
from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive'; OUT = f'{D}/Universal-ACMOT-Results/t4_gate'; os.makedirs(OUT, exist_ok=True)
%cd /content
!rm -rf /content/uacmot && git clone -q --branch universal-adapters-v1 https://github.com/AhmedCode110/AC-MOT.git /content/uacmot
%cd /content/uacmot
TARGET = open('research/T4_TARGET_COMMIT.txt').read().split()[0]
!git checkout -q $TARGET && git log --oneline -1
assert os.path.exists('configs/universal_acmot_policy_v5.json'), 'candidate V5 config missing at target commit'
!pip -q install ultralytics==8.3.200 lap==0.5.12

In [ ]:
os.makedirs('weights', exist_ok=True)
!test -f yolov8n.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt
!test -f rtdetr-l.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/rtdetr-l.pt
!test -f weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth || wget -q -P weights https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
EXPECT = {'yolov8n.pt':'f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36',
          'rtdetr-l.pt':'6de60b10d4bc566f00cda0f5b4d64afe4b66d48dc9695d2171effb7859d8e73f',
          'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth':'dd69338a24b8d7381807e247652bdc356325bcbaf1cd3e092e00e0a1a58706bf'}
w_ok = {p: hashlib.sha256(open(p,'rb').read()).hexdigest() == h for p, h in EXPECT.items()}; print(w_ok); assert all(w_ok.values())

In [ ]:
# Gate subset (Amendment 5f) + one val sequence (detection-level FRCNN check and benchmark clip)
from tools.fidelity_gate import GATE
TR = f'{D}/AC-MOT-shared/AC-MOT-data/VisDrone2019-MOT-train'; VA = f'{D}/AC-MOT-shared/AC-MOT-data/VisDrone2019-MOT-val'
for s in GATE:
    shutil.copytree(f'{TR}/sequences/{s}', f'/content/gate/sequences/{s}', dirs_exist_ok=True)
shutil.copytree(f'{VA}/sequences/uav0000137_00458_v', '/content/valclip/sequences/uav0000137_00458_v', dirs_exist_ok=True)
FR = 'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth'
!PYTHONPATH=. python -u tools/cache_detections.py --weights yolov8n.pt --dataset /content/gate --output-dir "$OUT/train/yolov8" --resolutions 736 2>&1 | grep -v -i warn
!PYTHONPATH=. python -u tools/cache_detections.py --weights rtdetr-l.pt --dataset /content/gate --output-dir "$OUT/train/rtdetr" --resolutions 736 2>&1 | grep -v -i warn
!PYTHONPATH=. python -u tools/cache_detections.py --weights $FR --dataset /content/valclip --output-dir "$OUT/val/fasterrcnn" --resolutions 640 736 2>&1 | grep -v -i warn

In [ ]:
# OFFICIAL T4 timing of the target commit (V4 and V5 policy files)
!PYTHONPATH=. python tools/t4_benchmark.py --dataset /content/valclip --sequence uav0000137_00458_v \
   --weights yolov8n.pt rtdetr-l.pt $FR --levels 640 736 832 --trackers bytetrack botsort \
   --frames 200 --warmup 20 --policy-files configs/universal_acmot_policy_v4.json configs/universal_acmot_policy_v5.json \
   --out "$OUT/t4_benchmark_{TARGET[:10]}.json" 2>&1 | grep -v -i warn
!pip freeze | grep -i -E "torch|ultralytics|numpy|opencv|lap" > "$OUT/packages.txt"
json.dump(dict(commit=TARGET, gpu=torch.cuda.get_device_name(0), torch=torch.__version__, weights_ok=w_ok,
               npz={p.replace(OUT+'/', ''): len(glob.glob(p+'/*.npz')) for p in glob.glob(f'{OUT}/*/*')}),
          open(f'{OUT}/T4_GATE_DONE.json','w'), indent=1)
print(open(f'{OUT}/T4_GATE_DONE.json').read())